# ViFinQA — BGE-M3 rows-first held-out ablation V1

Research-only evaluation of `passage_layout=rows_first` at `max_seq_length=384` on synthetic issuer-held-out validation/test rows. The notebook verifies the immutable source snapshot, applies one hash-pinned minimal patch, and never promotes a model.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
os.environ['WANDB_DISABLED']='true'; os.environ['WANDB_MODE']='disabled'
REPO_DIR=Path('/kaggle/working/AI_guru_rows_first_ablation')
SOURCE_TREE='ai_guru_synthetic_retriever_source_v1'; SOURCE_MANIFEST='ai_guru_synthetic_retriever_source_v1.manifest.json'
source_dirs=sorted({p.parent for p in Path('/kaggle/input').rglob(SOURCE_MANIFEST) if (p.parent/SOURCE_TREE).is_dir()})
if len(source_dirs)!=1: raise RuntimeError(f'Expected one verified source snapshot, found {len(source_dirs)}')
source_dir=source_dirs[0]; manifest=json.loads((source_dir/SOURCE_MANIFEST).read_text()); source_root=source_dir/SOURCE_TREE
files={p.relative_to(source_root).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in source_root.rglob('*') if p.is_file() and p.relative_to(source_root).as_posix()!='pax_global_header'}
tree_sha=hashlib.sha256(json.dumps(files,sort_keys=True,separators=(',',':')).encode()).hexdigest()
if manifest.get('protocol')!='kaggle_synthetic_retriever_source_v1' or tree_sha!=manifest.get('source_tree_sha256') or len(files)!=manifest.get('source_tree_file_count'): raise ValueError('Source snapshot hash failed')
if not str(manifest.get('git_commit','')).startswith('fde3629'): raise ValueError('Unexpected source snapshot revision')
if REPO_DIR.exists(): raise RuntimeError(f'Refusing to overwrite {REPO_DIR}')
shutil.copytree(source_root,REPO_DIR)
PATCH_NAME='rows_first_source.patch'; PATCH_SHA='6fce23804308cc5952f435b9efb0e9df866dbe7b272b20b2aa769b264b67acb2'
patches=sorted(Path('/kaggle/working').rglob(PATCH_NAME))
if len(patches)!=1: raise RuntimeError(f'Expected one {PATCH_NAME}, found {len(patches)}')
patch=patches[0]
if hashlib.sha256(patch.read_bytes()).hexdigest()!=PATCH_SHA: raise ValueError('Rows-first patch hash failed')
subprocess.run(['git','apply','--check',str(patch)],cwd=REPO_DIR,check=True); subprocess.run(['git','apply',str(patch)],cwd=REPO_DIR,check=True)
evaluator=REPO_DIR/'scripts/evaluate_synthetic_retriever_v1.py'; curriculum_source=REPO_DIR/'src/finance_query/synthetic_curriculum.py'
if hashlib.sha256(evaluator.read_bytes()).hexdigest()!='a11c215981b9bda0dba8d59b7f3d6e40b4cc5609cc25e8d1636349344cdc98d6': raise ValueError('Patched evaluator hash failed')
if hashlib.sha256(curriculum_source.read_bytes()).hexdigest()!='c414bea55348c20a53eca6a597c513c6654af09b81a9bf2ff9b5d63578f4825f': raise ValueError('Patched curriculum source hash failed')
print({'source_commit':manifest['git_commit'],'source_tree_sha256':tree_sha,'patch_sha256':PATCH_SHA})

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','torch==2.12.1','torchvision==0.27.1','--index-url','https://download.pytorch.org/whl/cu126'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','sentence-transformers==3.4.1','transformers==4.48.3'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),'--no-deps'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Enable Kaggle GPU and restart.')
vram_gib=torch.cuda.get_device_properties(0).total_memory/1024**3
if vram_gib<14: raise RuntimeError(f'Requires >=14 GiB VRAM; found {vram_gib:.1f} GiB')
print({'gpu':torch.cuda.get_device_name(0),'vram_gib':round(vram_gib,2),'torch':torch.__version__})

In [ ]:
INPUT_ROOT=Path('/kaggle/input'); CURRICULUM_NAME='synthetic_finance_curriculum_v1.jsonl'; MANIFEST_NAME='synthetic_finance_curriculum_v1.manifest.json'
curriculum_dirs=sorted({p.parent for p in INPUT_ROOT.rglob(CURRICULUM_NAME) if (p.parent/MANIFEST_NAME).is_file()})
if len(curriculum_dirs)!=1: raise RuntimeError(f'Expected one curriculum input, found {len(curriculum_dirs)}')
CURRICULUM_DIR=curriculum_dirs[0]; CURRICULUM=CURRICULUM_DIR/CURRICULUM_NAME; CURRICULUM_MANIFEST=CURRICULUM_DIR/MANIFEST_NAME
tables=sorted(INPUT_ROOT.rglob('table_assets.jsonl'))
if len(tables)!=1: raise RuntimeError(f'Expected one table_assets.jsonl, found {len(tables)}')
TABLES=tables[0]
model_dirs=sorted({p.parent for p in INPUT_ROOT.rglob('training_metadata.json') if (p.parent/'model.safetensors').is_file() and (p.parent/'modules.json').is_file()})
if len(model_dirs)!=1: raise RuntimeError(f'Expected one trained model output, found {len(model_dirs)}')
FINETUNED_MODEL=model_dirs[0]; training_metadata=json.loads((FINETUNED_MODEL/'training_metadata.json').read_text())
if training_metadata.get('provenance')!='synthetic_execution_verified': raise ValueError('Unexpected trained-model provenance')
print({'curriculum':str(CURRICULUM),'tables':str(TABLES),'finetuned_model':str(FINETUNED_MODEL)})

## Controlled ablation

Only passage order changes. The evaluator remains issuer-held-out, hash-bound, and non-promotable.

In [ ]:
OUTPUT_DIR=Path('/kaggle/working/bge_m3_rows_first_heldout_ablation_v1')
command=[sys.executable,str(evaluator),'--curriculum',str(CURRICULUM),'--manifest',str(CURRICULUM_MANIFEST),'--bundle-tables',str(TABLES),'--output-dir',str(OUTPUT_DIR),'--model','base=BAAI/bge-m3','--model',f'finetuned={FINETUNED_MODEL}','--splits','validation','test','--ks','1','3','5','10','20','--passage-batch-size','16','--query-batch-size','32','--max-seq-length','384','--passage-layout','rows_first','--device','cuda:0']
print('Running:', ' '.join(command)); subprocess.run(command,cwd=REPO_DIR,check=True)

In [ ]:
result=json.loads((OUTPUT_DIR/'evaluation_manifest.json').read_text())
if result.get('promotion_status')!='offline_evaluation_complete_not_promoted': raise ValueError('Unexpected promotion status')
if result.get('configuration',{}).get('passage_layout')!='rows_first': raise ValueError('Rows-first layout not recorded')
summary={label:{split:{'mrr':round(m['mrr'],4),'recall@10':round(m['recall_at_k']['10'],4),'top1_errors':m['top1_error_counts']} for split,m in model_result['splits'].items()} for label,model_result in result['models'].items()}
print(json.dumps({'summary':summary,'status':result['promotion_status'],'artifacts':sorted(p.name for p in OUTPUT_DIR.iterdir())},ensure_ascii=False,indent=2))

## Decision boundary

Use this artifact only to compare against the prior context-first baseline. Accept rows-first for later training only if both held-out splits improve and wrong-year/wrong-scope rates do not worsen.